# Data Explorer Agent
### Built on NVIDIA NeMo Agent Toolkit

---

**Data Explorer** is an AI-powered agent for automated data analysis. The full pipeline (learn → distill → inference) achieved **#1 on [DABStep](https://huggingface.co/datasets/adyen/DABstep)** (Data Agent Benchmark for Multi-step Reasoning), outperforming solutions from AntGroup, Google AI, and the Claude Code baseline.

**This launchable demonstrates the inference phase from scratch** -- a tool-calling agent reasoning over the data files with no pre-distilled domain knowledge. To reproduce the leaderboard configuration, run the full learn → distill workflow locally (see `dabstep_agent/learn/`).

Built by the **KGMON-LLM Agent Research Team** at NVIDIA:
- Jiwei Liu -- Team Lead, DABStep architecture
- Maximilian Jeblick -- Nemotron Super testing & optimization
- Jack Yu -- EDA workflow & multi-turn chat
- Alessio -- Generic QA agent


## Setup

Run the two cells below to install dependencies and enter your API keys. The first cell takes ~30 seconds on first run (instant on subsequent runs). The second cell prompts for API keys -- you only need the key for the demo you want to run:

| Key | Demo | Get one at |
|-----|------|-----------|
| `ANTHROPIC_API_KEY` | DABStep + QA demos (Claude Haiku) | https://console.anthropic.com |
| `OPENAI_API_KEY` | EDA demo (GPT-5 mini) | https://platform.openai.com |

In [ ]:
import subprocess, sys, os, shutil, pathlib

# Find repo root reliably (works regardless of kernel CWD)
repo_root = None

# Try git first
_result = subprocess.run(['git', 'rev-parse', '--show-toplevel'], capture_output=True, text=True)
if _result.returncode == 0:
    repo_root = _result.stdout.strip()

# Fallback: search upward from CWD for pyproject.toml
if not repo_root:
    _p = pathlib.Path.cwd()
    for _dir in [_p] + list(_p.parents):
        if (_dir / 'pyproject.toml').exists():
            repo_root = str(_dir)
            break

# Fallback: check known locations (Brev, Docker)
if not repo_root:
    for _candidate in [pathlib.Path.home() / 'data-explorer-launchable', pathlib.Path('/app')]:
        if (_candidate / 'pyproject.toml').exists():
            repo_root = str(_candidate)
            break

if not repo_root:
    raise RuntimeError('Could not find repo root -- set repo_root manually')

os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# --- Step 1: Install project dependencies via uv (for the server) ---
uv_bin = shutil.which('uv') or os.path.expanduser('~/.local/bin/uv')

print('Installing project dependencies (instant if already done)...')
result = subprocess.run([uv_bin, 'sync', '--all-groups', '--frozen'],
                        cwd=repo_root, capture_output=True, text=True)
if result.returncode != 0:
    print(f'ERROR: uv sync failed:\n{result.stderr}')
    raise RuntimeError('Dependency installation failed')
print('  Project dependencies OK')

# --- Step 2: Install notebook dependencies into the running kernel ---
print('Installing notebook dependencies...')
result = subprocess.run(
    [uv_bin, 'pip', 'install', '--quiet', '--python', sys.executable,
     'pandas', 'numpy', 'matplotlib', 'seaborn', 'requests', 'python-dotenv'],
    capture_output=True, text=True)
if result.returncode != 0:
    print(f'ERROR: uv pip install failed:\n{result.stderr}')
    raise RuntimeError('Notebook dependency installation failed')
print('  Notebook dependencies OK')

# --- Step 3: Download DABStep data if missing ---
if not os.path.exists(os.path.join(repo_root, 'data', 'context', 'payments.csv')):
    print('Downloading DABStep data from HuggingFace (~30s)...')
    result = subprocess.run([uv_bin, 'run', 'python', 'dabstep_agent/download_data.py'],
                            cwd=repo_root, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'ERROR: Data download failed:\n{result.stderr}')
        raise RuntimeError('Data download failed')
    print('  Data downloaded')
else:
    print('  Data already present')

# --- Step 4: Generate file_structures.json cache if missing ---
if not os.path.exists(os.path.join(repo_root, 'data', 'context', 'file_structures.json')):
    print('Generating data schema cache...')
    subprocess.run([uv_bin, 'run', 'python', 'dabstep_agent/generate_file_structures.py'],
                   cwd=repo_root, capture_output=True, text=True, check=True)
    print('  Schema cache generated')
else:
    print('  Schema cache already present')

print('\nSetup complete.')

In [ ]:
import getpass, os

keys = {
    'ANTHROPIC_API_KEY': {
        'prefix': 'sk-ant-',
        'demo': 'DABStep + QA demos',
        'url': 'https://console.anthropic.com',
    },
    'OPENAI_API_KEY': {
        'prefix': 'sk-',
        'demo': 'EDA demo',
        'url': 'https://platform.openai.com',
    },
}

print('API Key Setup')
print('(press Enter to skip a key you don\'t need)\n')

for var, info in keys.items():
    current = os.environ.get(var, '')
    if current.startswith(info['prefix']):
        print(f'  {var}: already set  -->  {info["demo"]}')
    else:
        key = getpass.getpass(f'  Enter {var} (for {info["demo"]}, or Enter to skip): ')
        if key.startswith(info['prefix']):
            os.environ[var] = key
            print(f'    Set successfully')
        elif key:
            print(f'    WARNING: expected key starting with {info["prefix"]!r} -- skipped')
        else:
            print(f'    Skipped  -->  {info["demo"]} will not work')
            print(f'    Get a key at {info["url"]}')
    print()

# Save keys to secrets.env so demo notebooks can load them
_env_path = os.path.join(repo_root, 'secrets.env')
_lines = []
for var in keys:
    val = os.environ.get(var, '')
    if val:
        _lines.append(f'{var}={val}')
if _lines:
    with open(_env_path, 'w') as f:
        f.write('\n'.join(_lines) + '\n')
    print(f'Keys saved to secrets.env (loaded automatically by demo notebooks)')

## Published Results

These results come from running the full learn → distill → inference pipeline. **The launchable runs only the inference phase from scratch** (no distilled artifacts), so quality on this launchable will be lower than the numbers below. Reproduce these by running `dabstep_agent/learn/` locally first.

| System | Hard Accuracy | Time/Task | Code Length |
|--------|:---:|:---:|:---:|
| **Data Explorer + Haiku 4.5 (Ours)** | **89.95%** | **20s** | **1,870** |
| Data Explorer + Nemotron 3 Super (Ours) | ~77% | 20s | 1,870 |
| DataPilot (AntGroup) | 87.57% | -- | -- |
| Claude Code + Opus 4.5 (baseline) | 66.93% | 10 min | 5,011 |
| DS-STAR (Google AI) | 45.24% | -- | -- |

**30x faster** than the baseline. **+23 points** on hard questions. Less than half the code.


## How It Works: Learn, Infer, Reflect

The key innovation is a **three-phase architecture** that separates expensive learning from fast inference:

```
Phase 1: LEARNING (offline)          Phase 2: INFERENCE (fast)          Phase 3: REFLECTION (offline)
┌──────────────────────┐            ┌──────────────────────┐           ┌──────────────────────┐
│  Heavy Model          │            │  Lightweight Model    │           │  Heavy Model          │
│  (Claude Opus 4.5)    │            │  (Claude Haiku)       │           │  (Claude Sonnet)      │
│                       │   distill  │                       │   review  │                       │
│  Solves training      │──────────▶ │  Uses distilled code  │─────────▶ │  Self-consistency     │
│  tasks, writes        │            │  + few-shot examples  │           │  checks, feeds back   │
│  reusable functions   │            │  to answer in seconds │           │  improved prompts     │
└──────────────────────┘            └──────────────────────┘           └──────────────────────┘
     Output:                              Output:                            Output:
     helper.py (22 functions)             answer + reasoning trace           improved Phase 2 prompts
     new_solutions.md (few-shot)          
```

**Key insight**: By investing upfront in learning and code distillation, even a small, fast model can outsmart heavier models on complex multi-step problems.

This launchable demonstrates **Phase 2 (Inference)** -- the distilled knowledge is already baked in.

## Quick Demo: Ask a Question

The cell below starts the DABStep inference server (port 8000), waits for it to be ready, then we ask it questions about the financial payments dataset (138K transactions, 1000 fee rules, 5 merchants).

Requires `ANTHROPIC_API_KEY` -- if not set, the demo cells will be skipped with a message.

In [ ]:
import subprocess, requests, time

_server_ready = False

if not os.environ.get('ANTHROPIC_API_KEY'):
    print('ANTHROPIC_API_KEY not set -- server will not start.')
    print('Get a key at https://console.anthropic.com')
    print('Then re-run the API Key Setup cell above.')
else:
    # Check if server is already running (Docker/Brev case)
    try:
        if requests.get('http://localhost:8000/docs', timeout=3).status_code == 200:
            _server_ready = True
            print('DABStep server already running at http://localhost:8000')
    except Exception:
        pass

    if not _server_ready:
        print('Starting DABStep server in background...')
        subprocess.Popen(
            [uv_bin, 'run', 'python', 'dabstep_agent/inference/server.py', '--port', '8000'],
            cwd=repo_root, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        for attempt in range(10):
            try:
                if requests.get('http://localhost:8000/docs', timeout=5).status_code == 200:
                    _server_ready = True
                    print(f'DABStep server is ready (took ~{(attempt + 1) * 5}s)')
                    break
            except requests.ConnectionError:
                pass
            if attempt < 9:
                print(f'  Waiting for server... ({attempt + 1}/10)')
                time.sleep(5)

        if not _server_ready:
            print('WARNING: Server did not start within 50s.')
            print('Try starting it manually: uv run python dabstep_agent/inference/server.py &')

In [ ]:
if not os.environ.get('ANTHROPIC_API_KEY'):
    print('ANTHROPIC_API_KEY not set -- skipping DABStep demo.')
    print('Get a key at https://console.anthropic.com')
elif not _server_ready:
    print('Server is not running -- skipping. Check the server startup cell above.')
else:
    def ask(question, guidelines='N/A'):
        """Send a question to the DABStep inference server."""
        t0 = time.time()
        try:
            resp = requests.post('http://localhost:8000/solve',
                                 json={'question': question, 'guidelines': guidelines}, timeout=300)
            resp.raise_for_status()
        except requests.ConnectionError:
            print('ERROR: Cannot connect to server at localhost:8000.')
            print('Try restarting: uv run python dabstep_agent/inference/server.py &')
            return None, 0, []
        except requests.exceptions.RequestException as e:
            print(f'ERROR: Request failed: {e}')
            return None, 0, []
        elapsed = time.time() - t0
        r = resp.json()
        return r['agent_answer'], round(elapsed, 1), r['reasoning_trace']

    result = ask('Which issuing country has the highest number of transactions?')
    if result[0] is not None:
        answer, secs, trace = result
        print(f'Answer:  {answer}')
        print(f'Time:    {secs}s')
        print(f'Steps:   {len([m for m in trace if m.get("role") == "ai" and m.get("tool_calls")])} tool calls')

In [ ]:
if os.environ.get('ANTHROPIC_API_KEY') and _server_ready:
    # Try a harder multi-step question
    result = ask(
        'For the 12th of the year 2023, what is the total fees (in euros) '
        'that Belles_cookbook_store should pay?'
    )
    if result[0] is not None:
        answer, secs, trace = result
        print(f'Answer:   {answer}')
        print(f'Expected: 12.91')
        print(f'Time:     {secs}s')

## Try It Yourself

Edit the question below and run the cell. The dataset has info on merchants, card schemes, fees, fraud, and transactions across 2023.

In [ ]:
if not os.environ.get('ANTHROPIC_API_KEY'):
    print('ANTHROPIC_API_KEY not set -- set it to try interactive questions.')
elif not _server_ready:
    print('Server is not running -- check the server startup cell above.')
else:
    # ---- Edit your question here ----
    my_question = 'What is the average transaction amount by card scheme?'
    # ----------------------------------

    result = ask(my_question)
    if result[0] is not None:
        answer, secs, _ = result
        print(f'Answer: {answer}  ({secs}s)')

## Explore the Full Demos

This launchable includes three complete demo notebooks:

| Notebook | What it shows | Model | API Key |
|----------|--------------|-------|--------|
| [demo_dabstep.ipynb](demo_dabstep.ipynb) | **DABStep inference** -- agent solving from scratch | Claude Haiku (Anthropic) | `ANTHROPIC_API_KEY` |
| [demo_qa.ipynb](demo_qa.ipynb) | **General tabular QA** -- works on any dataset, no prior knowledge needed | Claude Haiku (Anthropic) | `ANTHROPIC_API_KEY` |
| [demo_eda.ipynb](demo_eda.ipynb) | **Open-ended EDA** -- generates a full analysis notebook from any CSV | GPT-5 mini (OpenAI) | `OPENAI_API_KEY` |


## Architecture

```
┌────────────────────────────────────────────────────────────┐
│                    NVIDIA Launchable                       │
│                                                            │
│  JupyterLab (port 8888)                                   │
│    ├── START_HERE.ipynb     <- you are here               │
│    ├── demo_dabstep.ipynb   -> FastAPI /solve endpoint     │
│    ├── demo_qa.ipynb        -> Generic QA (any dataset)    │
│    └── demo_eda.ipynb       -> EDA notebook generator      │
│                                                            │
│  DABStep Server (port 8000)                                │
│    └── POST /solve -> Tool-calling Agent                   │
│                      └── python_executor (pandas, numpy)  │
│                                                            │
│  Data: payments.csv, fees.json, merchant_data.json         │
└────────────────────────────────────────────────────────────┘
          │                              │
          ▼                              ▼
   Anthropic API                    OpenAI API
   (Claude Haiku)                   (GPT-5 mini)
   ANTHROPIC_API_KEY                OPENAI_API_KEY
   DABStep + QA demos               EDA demo
```